# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique:LoRA (Low-Rank Adaptation) - trains only a small fraction of model parameters
* Model:  GPT-2
* Evaluation approach: Accuracy score on validation set 
* Fine-tuning dataset: Emotion dataset - 6 emotional categories (sadness, joy, love, anger, fear, surprise)

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [50]:
# Install required packages
!pip install transformers datasets evaluate peft accelerate
! pip install -U scikit-learn

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate

# Check if we have GPU available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 40.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 36.9 MB/s eta 0:00:00
Using: cuda


In [51]:
# Load GPT-2 model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set padding token to: {tokenizer.pad_token}")

original_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label={0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'},
    label2id={'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}
)

original_model.config.pad_token_id = tokenizer.pad_token_id
original_model.to(device)
print("Loaded GPT-2 model with proper padding configuration")

Set padding token to: <|endoftext|>


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded GPT-2 model with proper padding configuration


In [52]:
# Load emotion dataset
dataset = load_dataset("emotion")
print(f"Dataset loaded: {len(dataset['train'])} training examples, {len(dataset['validation'])} validation examples")

print("\n First 2 examples:")
for i in range(2):
    text = dataset['train']['text'][i]
    label = dataset['train']['label'][i]
    emotion = original_model.config.id2label[label]
    print(f"Text: {text}")
    print(f"Emotion: {emotion}\n")

Dataset loaded: 16000 training examples, 2000 validation examples

 First 2 examples:
Text: i didnt feel humiliated
Emotion: sadness

Text: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
Emotion: sadness



In [53]:
def tokenize_examples(examples):
    return tokenizer(
        examples["text"], 
        padding=False,
        truncation=True, 
        max_length=256
    )

tokenized_datasets = dataset.map(tokenize_examples, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
print("Data tokenized and ready")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Data tokenized and ready


In [54]:
# Evaluation function
def calculate_accuracy(eval_pred):
    accuracy_metric = evaluate.load("accuracy")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Test base model performance
eval_trainer = Trainer(
    model=original_model,
    args=TrainingArguments(
        output_dir="./eval_results",
        per_device_eval_batch_size=8,
        report_to=None
    ),
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=calculate_accuracy,
)

print("Testing base model...")
base_results = eval_trainer.evaluate()
base_accuracy = base_results['eval_accuracy']
print(f"Base model accuracy: {base_accuracy:.4f}")

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Testing base model...


Base model accuracy: 0.2850


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [56]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["c_attn"]
)

# Convert to PEFT model
peft_model = get_peft_model(original_model, lora_config)
peft_model.to(device)
print("PEFT model ready")
peft_model.print_trainable_parameters()

PEFT model ready
trainable params: 589,824 || all params: 125,038,848 || trainable%: 0.471712599271548


In [57]:
training_args = TrainingArguments(
    output_dir="/tmp/emotion-classifier",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=None,
    logging_steps=50,
)


In [58]:
# Create trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=calculate_accuracy,
)

# Start training
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.338500,0.223899,0.918500
2,0.184900,0.159445,0.929000
3,0.107200,0.119966,0.941500


TrainOutput(global_step=6000, training_loss=0.3709859150250753, metrics={'train_runtime': 344.674, 'train_samples_per_second': 139.262, 'train_steps_per_second': 17.408, 'total_flos': 959113059655680.0, 'train_loss': 0.3709859150250753, 'epoch': 3.0})

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [66]:
# Saving the model
trainer.save_model("/tmp/emotion-classifier-final")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [60]:
tuned_model = PeftModel.from_pretrained(
    original_model,
    "/tmp/emotion-classifier-final",
    torch_dtype=torch.float32
)
tuned_model.to(device)
print("Fine-tuned model loaded")

Fine-tuned model loaded


In [61]:
# Evaluate the improved model
test_trainer = Trainer(
    model=tuned_model,
    args=TrainingArguments(
        output_dir="./test_results",
        per_device_eval_batch_size=8,
        report_to=None
    ),
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=calculate_accuracy,
)

print("Testing fine-tuned model...")
tuned_results = test_trainer.evaluate()
tuned_accuracy = tuned_results['eval_accuracy']
print(f"Fine-tuned model accuracy: {tuned_accuracy:.4f} ({tuned_accuracy*100:.1f}%)")

Testing fine-tuned model...


Fine-tuned model accuracy: 0.9415 (94.2%)


In [62]:
# Compare results ✓
print("\n" + "="*50)
print("RESULTS COMPARISON")
print("="*50)
print(f"Base model accuracy:    {base_accuracy:.4f} ({base_accuracy*100:.1f}%)")
print(f"Fine-tuned accuracy:    {tuned_accuracy:.4f} ({tuned_accuracy*100:.1f}%)")
improvement = tuned_accuracy - base_accuracy
print(f"Improvement:           +{improvement:.4f} (+{improvement*100:.1f}%)")
print("="*50)

if improvement > 0:
    print("✓ Training successful - model improved!")
else:
    print("✗ Training didn't improve performance")


RESULTS COMPARISON
Base model accuracy:    0.2850 (28.5%)
Fine-tuned accuracy:    0.9415 (94.2%)
Improvement:           +0.6565 (+65.7%)
✓ Training successful - model improved!


In [64]:
 def classify_emotion(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=1).item()
        confidence = predictions[0][predicted_class].item()
    
    return model.config.id2label[predicted_class], confidence

# Test examples
test_sentences = [
    "I'm thrilled about the good news!",
    "This situation is making me very nervous",
    "I'm so angry about what happened",
    "Feeling romantic and affectionate tonight",
    "That surprise party was amazing!",
    "I feel completely alone and hopeless"
]

print("\n" + "="*60)
print("SIDE-BY-SIDE COMPARISON")
print("="*60)

for i, sentence in enumerate(test_sentences):
    base_emotion, base_conf = classify_emotion(sentence, original_model, tokenizer)
    tuned_emotion, tuned_conf = classify_emotion(sentence, tuned_model, tokenizer)
    
    print(f"\nTest {i+1}: '{sentence}'")
    print(f"  Base model:  {base_emotion:8} ({base_conf:.3f})")
    print(f"  Tuned model: {tuned_emotion:8} ({tuned_conf:.3f})")
    
    if base_emotion != tuned_emotion:
        print(f"  → Prediction changed!")


SIDE-BY-SIDE COMPARISON

Test 1: 'I'm thrilled about the good news!'
  Base model:  joy      (0.921)
  Tuned model: joy      (0.921)

Test 2: 'This situation is making me very nervous'
  Base model:  fear     (0.886)
  Tuned model: fear     (0.886)

Test 3: 'I'm so angry about what happened'
  Base model:  anger    (0.965)
  Tuned model: anger    (0.965)

Test 4: 'Feeling romantic and affectionate tonight'
  Base model:  love     (0.820)
  Tuned model: love     (0.820)

Test 5: 'That surprise party was amazing!'
  Base model:  joy      (0.929)
  Tuned model: joy      (0.929)

Test 6: 'I feel completely alone and hopeless'
  Base model:  sadness  (1.000)
  Tuned model: sadness  (1.000)


In [65]:
# Final summary
print("\n" + "="*70)
print("PROJECT COMPLETE: Lightweight Fine-Tuning Results")
print("="*70)
print(f"✓ Base model accuracy:    {base_accuracy*100:.1f}%")
print(f"✓ Tuned model accuracy:   {tuned_accuracy*100:.1f}%")
print(f"✓ Improvement:           +{improvement*100:.1f}%")
print(f"✓ Efficiency:            {efficiency:.2f}% parameters trained")
print(f"✓ Model size:            {total_params:,} total parameters")
print(f"✓ Storage:               Saved to /tmp/ for workspace safety")
print("\nThe project successfully demonstrated parameter-efficient")
print("fine-tuning using LoRA, achieving better performance while")
print("training only a small fraction of the model's parameters.")
print("="*70)


PROJECT COMPLETE: Lightweight Fine-Tuning Results
✓ Base model accuracy:    28.5%
✓ Tuned model accuracy:   94.2%
✓ Improvement:           +65.7%
✓ Efficiency:            0.00% parameters trained
✓ Model size:            125,038,848 total parameters
✓ Storage:               Saved to /tmp/ for workspace safety

The project successfully demonstrated parameter-efficient
fine-tuning using LoRA, achieving better performance while
training only a small fraction of the model's parameters.
